In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import us

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Impact of Crime Rates and Air Quality on Housing Prices in the United States

## 1. Introduction

The real estate market is influenced by a wide range of factors, including economic conditions, demographic characteristics, and environmental quality. In recents years, there has been growing interest in understanding how non-traditional factors such as crime rates and air pollution impact housing prices.

This project aims to explore the relationship between housing prices and environmental conditions in the United States by combining multiple data sources, including housing data, crime statistics and air quality measurements. By integrating these datasets, the analysis provides a more comprehensive view of the factors that influence properly values.

In addition to traditionalhousing features such as income, population, and property characteristics, this study incorporates crime-related variables and air pollution indicators to evaluate their impact on real estate prices.

Machine learning techniques are applied to model these relathioships and assess the predictive power of environmental and socioeconomic variables. The results aim to hghlight whether environmental quality plays a significant role in housing valuation and how it compares to more conventional factors.

## 2. Data Overview

This project combines multiple datasets to analyze housing prices in relation to environmental and socioeconomic factors.

The main datasets used include:
* A housing dataset containg property-related features such as income, number of rooms, house age, price and population. => https://www.kaggle.com/datasets/vedavyasv/usa-housing?resource=download
* A crime dataset providing information aboout different types of crimes, including violent crime and property crime. => https://www.kaggle.com/datasets/kabhishm/united-states-crime-rates-by-city-population?select=crime_60_100.csv
* An air quality dataset containing poluution indicators such as NO2, O3, SO2 and CO levels. => https://www.kaggle.com/datasets/sogun3/uspollution

These dataset were merged based on geographical information (state-level data), allowing the integration of multiple independent data sources into a unified dataset.

The final dataset includes both numerical and categorical variables and represents multiple aspects of the living environment.

# 3. Data Processing

### 3.1 Housing Data

In [2]:
housing_data = pd.read_csv("data/USA_Housing.csv")

In [3]:
housing_data.head(10)

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."
2,61287.067179,5.865890,8.512727,5.13,36882.159400,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482..."
3,63345.240046,7.188236,5.586729,3.26,34310.242831,1.260617e+06,USS Barnett\nFPO AP 44820
4,59982.197226,5.040555,7.839388,4.23,26354.109472,6.309435e+05,USNS Raymond\nFPO AE 09386
5,80175.754159,4.988408,6.104512,4.04,26748.428425,1.068138e+06,"06039 Jennifer Islands Apt. 443\nTracyport, KS..."
6,64698.463428,6.025336,8.147760,3.41,60828.249085,1.502056e+06,"4759 Daniel Shoals Suite 442\nNguyenburgh, CO ..."
7,78394.339278,6.989780,6.620478,2.42,36516.358972,1.573937e+06,"972 Joyce Viaduct\nLake William, TN 17778-6483"
8,59927.660813,5.362126,6.393121,2.30,29387.396003,7.988695e+05,USS Gilbert\nFPO AA 20957
9,81885.927184,4.423672,8.167688,6.10,40149.965749,1.545155e+06,Unit 9446 Box 0958\nDPO AE 97025


In [4]:
housing_data.shape

(5000, 7)

In [5]:
housing_data.isnull().sum()

Avg. Area Income                0
Avg. Area House Age             0
Avg. Area Number of Rooms       0
Avg. Area Number of Bedrooms    0
Area Population                 0
Price                           0
Address                         0
dtype: int64

In [6]:
housing_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Avg. Area Income              5000 non-null   float64
 1   Avg. Area House Age           5000 non-null   float64
 2   Avg. Area Number of Rooms     5000 non-null   float64
 3   Avg. Area Number of Bedrooms  5000 non-null   float64
 4   Area Population               5000 non-null   float64
 5   Price                         5000 non-null   float64
 6   Address                       5000 non-null   str    
dtypes: float64(6), str(1)
memory usage: 273.6 KB


In [7]:
housing_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Avg. Area Income,5000.0,6.858311e+04,10657.991214,17796.631190,61480.562388,6.880429e+04,7.578334e+04,1.077017e+05
Avg. Area House Age,5000.0,5.977222e+00,0.991456,2.644304,5.322283,5.970429e+00,6.650808e+00,9.519088e+00
Avg. Area Number of Rooms,5000.0,6.987792e+00,1.005833,3.236194,6.299250,7.002902e+00,7.665871e+00,1.075959e+01
Avg. Area Number of Bedrooms,5000.0,3.981330e+00,1.234137,2.000000,3.140000,4.050000e+00,4.490000e+00,6.500000e+00
Area Population,5000.0,3.616352e+04,9925.650114,172.610686,29403.928702,3.619941e+04,4.286129e+04,6.962171e+04
Price,5000.0,1.232073e+06,353117.626581,15938.657923,997577.135049,1.232669e+06,1.471210e+06,2.469066e+06


The housing dataset contains 5000 observations and 7 variables providing key information about residentail properties.

The dataset includes several numerical features such as average area income, house age, number of rooms, number of bedrooms and population, as well as the target variable - housing price. Additionally, it contains and address column which represents the location of each property but is not directly used in the modeling process.

All numerical variables are complete, with no missing values, making the dataset suitable for direct analysis. This features are represented as floating-point numbers, while address columns is stored as string.

Summary statistics show that the average area income is approximately 68 000 while the average house price is around 1.23 million. The number of rooms average close to 7, indicating relatively large residential properties.

The dataset provides a solid foundation for analyzing the relationship between housing characteristics and property prices before integrating additional external factors such as crime and air quality.

In [8]:
housing_data["State"] = housing_data["Address"].str.extract(r'\b([A-Z]{2})\s+\d{5}\b')
housing_data = housing_data[housing_data.State.str.isupper()]

As part of the data processing stage, geographical information was extracted from the address field in the sousing dataset. SInce the original dataset did not include a separate state columns, a regular expression was included to identify and extract the state abbreviation from the address string. Specifically, the state code was extracted by identifying patterns consisting of two uppercase letters followed by ZIP code. This allowed the creation of a new **State** feature, which was later used to merge the housing dataset with external datasets containing crime statistics and air quality data. 

In [9]:
valid_states = [state.abbr for state in us.states.STATES]
housing_data = housing_data[housing_data.State.isin(valid_states)]
housing_data.isnull().sum()

Avg. Area Income                0
Avg. Area House Age             0
Avg. Area Number of Rooms       0
Avg. Area Number of Bedrooms    0
Area Population                 0
Price                           0
Address                         0
State                           0
dtype: int64

To ensure data quality only valid state abbreviations were retained. This step was essential for aligning the datasets and enabling accurate integration based on geographical information.

### 3.2 Crime Data

In [10]:
crime_data_40_60 = pd.read_csv("data/crime_40_60.csv")
crime_data_40_60.head(5)

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft
0,Pennsylvania,"Abington Township, Montgomery County","55,731",197.4,1.8,14.4,70.0,111.2,1979.1,296.1,1650.8,32.3
1,Oregon,Albany,"51,084",86.1,0.0,19.6,45.0,21.5,3092.9,438.5,2470.4,184.0
2,Louisiana,Alexandria,"48,449",1682.2,18.6,28.9,293.1,1341.6,7492.4,2010.4,5102.3,379.8
3,California,Aliso Viejo,"48,999",87.8,0.0,0.0,12.2,75.5,847.0,208.2,612.3,26.5
4,Florida,Altamonte Springs,"42,296",335.7,2.4,21.3,82.8,229.3,3057.0,427.9,2463.6,165.5


In [11]:
crime_data_40_60.shape

(358, 12)

In [12]:
crime_data_40_60.info()

<class 'pandas.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         358 non-null    str    
 1   cities         358 non-null    str    
 2   population     358 non-null    str    
 3   violent_crime  350 non-null    float64
 4   murder         358 non-null    float64
 5   rape           352 non-null    float64
 6   robbery        358 non-null    float64
 7   agrv_assault   356 non-null    float64
 8   prop_crime     357 non-null    float64
 9   burglary       357 non-null    float64
 10  larceny        358 non-null    float64
 11  vehicle_theft  358 non-null    float64
dtypes: float64(9), str(3)
memory usage: 33.7 KB


In [13]:
crime_data_40_60.describe().T

,count,mean,std,min,25%,50%,75%,max
violent_crime,350.0,308.175714,286.503643,12.0,121.700,223.05,394.350,2362.1
murder,358.0,3.017598,5.081112,0.0,0.000,1.80,4.075,52.7
rape,352.0,25.881250,23.664427,0.0,8.450,19.05,37.200,178.7
robbery,358.0,84.647765,91.400726,0.0,26.225,54.60,112.850,779.9
agrv_assault,356.0,190.373315,204.143095,4.5,60.750,129.35,236.075,1856.9
prop_crime,357.0,2859.556863,1402.755755,650.5,1816.400,2592.00,3521.400,8225.2
burglary,357.0,606.110364,425.179055,69.9,304.700,478.00,779.200,2368.9
larceny,358.0,2074.473184,1014.901448,468.6,1331.625,1839.90,2608.250,5764.6
vehicle_theft,358.0,178.441341,160.102237,1.7,67.825,130.00,234.625,1118.4


In [14]:
crime_data_40_60.isnull().sum()

states           0
cities           0
population       0
violent_crime    8
murder           0
rape             6
robbery          0
agrv_assault     2
prop_crime       1
burglary         1
larceny          0
vehicle_theft    0
dtype: int64

The first crime dataset contains information about cities with population between 40 000 and 60 000 residents. It includes 358 observations and 12 variables, covering various types of crime such as violient crime, murder, rape, robbery, aggravated assault and property crime. Most variables are numeric and represent crime rates per population, allowing for meaningful comparisons across cities. A small number of missing values were identified in some columns. The dataset provides a detailed view of crime patterns in medium-sized cities and serves as an important components for analyzing how crime levels influence housing prices. 

In [15]:
crime_data_60_100 = pd.read_csv("data/crime_60_100.csv")
crime_data_60_100.head(5)

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft
0,California,Alameda,"75,467",212.0,1.3,11.91,106.0,92.8,"2,507.1",392.2,"1,723.9",390.9
1,Georgia,Albany,"78,512","1,035.5",5.1,34.4,285.3,710.7,"6,369.7","1,793.4","4,291.1",285.3
2,New York,Albany,"98,187",816.8,4.1,43.8,253.6,515.3,"4,420.1",903.4,"3,359.9",156.8
3,California,Alhambra,"84,469",176.4,-,2.4,78.1,95.9,"2,271.8",384.8,"1,585.2",301.9
4,Texas,Allen,"88,783",61.9,-,12.4,14.6,34.9,"1,612.9",242.2,"1,321.2",49.6


In [16]:
crime_data_60_100.shape

(305, 12)

In [17]:
crime_data_60_100.info()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         305 non-null    str    
 1   cities         305 non-null    str    
 2   population     305 non-null    str    
 3   violent_crime  295 non-null    str    
 4   murder         305 non-null    str    
 5   rape           295 non-null    str    
 6   robbery        305 non-null    float64
 7   agrv_assault   305 non-null    str    
 8   prop_crime     304 non-null    str    
 9   burglary       304 non-null    str    
 10  larceny        305 non-null    str    
 11  vehicle_theft  305 non-null    str    
dtypes: float64(1), str(11)
memory usage: 28.7 KB


In [18]:
crime_data_60_100.describe().T

,count,mean,std,min,25%,50%,75%,max
robbery,305.0,116.519344,116.787983,4.8,38.9,81.6,152.3,972.1


In [19]:
crime_data_60_100.isnull().sum()

states            0
cities            0
population        0
violent_crime    10
murder            0
rape             10
robbery           0
agrv_assault      0
prop_crime        1
burglary          1
larceny           0
vehicle_theft     0
dtype: int64

The second dataset focuses on cities with populations between 60 000 and 100 000 residents and contains 305 observations. Similar to the first dataset, it includes multiple crime-related variables such as violent crime, robbery and assult. Some columns were initially stored as strings instead of numerical values, requiring type conversion during processing. This dataset expands the analysis by incorporating larger urban areas and contributes to a more comprehensive understanding of crime distribution.

In [20]:
crime_data_100_250 = pd.read_csv("data/crime_100_250.csv")
crime_data_100_250.head(5)

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft
0,Texas,Abilene,"119,886",393.7,2.5,31.7,105.9,253.6,"3,664.3",865,"2,656.7",142.6
1,Ohio,Akron,"198,390",886.6,12.1,84.2,290.8,499.5,"5,057.7","1,728.4","2,965.9",363.4
2,Virginia,Alexandria,"145,892",166.6,-,6.2,94.6,65.8,"2,049.5",192.6,"1,633.4",223.5
3,Pennsylvania,Allentown,"119,334",547.2,12.6,45.3,313.4,176,"3,857.2","1,045.8","2,503.1",308.4
4,Texas,Amarillo,"196,576",650.1,5.1,56.0,141.4,447.7,"4,527.5","1,061.7","3,145.9",320


In [21]:
crime_data_100_250.shape

(212, 12)

In [22]:
crime_data_100_250.info()

<class 'pandas.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         212 non-null    str    
 1   cities         212 non-null    str    
 2   population     212 non-null    str    
 3   violent_crime  211 non-null    str    
 4   murder         212 non-null    str    
 5   rape           211 non-null    float64
 6   robbery        212 non-null    float64
 7   agrv_assault   212 non-null    str    
 8   prop_crime     212 non-null    str    
 9   burglary       212 non-null    str    
 10  larceny        212 non-null    str    
 11  vehicle_theft  212 non-null    str    
dtypes: float64(2), str(10)
memory usage: 20.0 KB


In [23]:
crime_data_100_250.describe().T

,count,mean,std,min,25%,50%,75%,max
rape,211.0,32.128436,25.697522,2.2,15.8,27.6,41.550,265.7
robbery,212.0,157.465094,124.355664,13.3,62.0,129.2,212.375,662.2


In [24]:
crime_data_100_250.isnull().sum()

states           0
cities           0
population       0
violent_crime    1
murder           0
rape             1
robbery          0
agrv_assault     0
prop_crime       0
burglary         0
larceny          0
vehicle_theft    0
dtype: int64

The third dataset contains crime data for cities with populations between 100 000 and 250 000 residents with total of 212 observations. The structure is consistent with previous datasets including similar crime indicators. The dataset requires preprocessing steps such as converting data types and handling a small number of missing values. Compared to smaller cities, crime rates in this dataset show greater variability, reflecting the complexity of larger urban environments. This dataset plays a key role in capturing crime dynamics in larger metropolitan areas,

In [25]:
crime_data_250_plus = pd.read_csv("data/crime_250_plus.csv")
crime_data_250_plus.head(5)

,states,cities,population,total_crime,murder,rape,robbery,agrv_assault,tot_violent_crime,burglary,larceny,vehicle_theft,tot_prop_crim,arson
0,Alabama,Mobile3,"248,431",6217.02,20.13,58.16,177.11,485.85,740.25,"1,216.84","3,730.21",506.78,"5,453.83",22.94
1,Alaska,Anchorage,"296,188",6640.04,9.12,132.01,262.67,799.49,"1,203.29",748.17,"3,619.66","1,047.98","5,415.82",20.93
2,Arizona,Chandler,"249,355",2589.08,2.01,52.13,56.95,148.38,259.47,314.41,"1,866.01",149.18,"2,329.61",NaN
3,Arizona,Gilbert,"242,090",1483.75,2.07,16.11,21.07,46.26,85.51,192.49,"1,137.59",55.76,"1,385.85",12.39
4,Arizona,Glendale,"249,273",5037.85,4.81,38.91,192.96,251.53,488.22,637.45,"3,426.36",466.56,"4,530.37",19.26


In [26]:
crime_data_250_plus = crime_data_250_plus.rename(columns={
    "tot_violent_crime" : "violent_crime",
    "tot_prop_crim" : "prop_crime"
})

In [27]:
crime_data_250_plus.shape

(100, 14)

In [28]:
crime_data_250_plus.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         100 non-null    str    
 1   cities         100 non-null    str    
 2   population     100 non-null    str    
 3   total_crime    100 non-null    float64
 4   murder         100 non-null    float64
 5   rape           96 non-null     float64
 6   robbery        100 non-null    float64
 7   agrv_assault   100 non-null    str    
 8   violent_crime  96 non-null     str    
 9   burglary       100 non-null    str    
 10  larceny        100 non-null    str    
 11  vehicle_theft  100 non-null    str    
 12  prop_crime     100 non-null    str    
 13  arson          92 non-null     float64
dtypes: float64(5), str(9)
memory usage: 11.1 KB


In [29]:
crime_data_250_plus.describe().T

,count,mean,std,min,25%,50%,75%,max
total_crime,100.0,4312.471900,1674.275944,1381.31,3029.7800,4091.500,5392.1750,8734.98
murder,100.0,11.618400,10.940627,0.72,4.1850,8.595,15.7025,66.07
rape,96.0,59.942083,28.818067,13.85,38.4650,56.570,75.4425,144.67
robbery,100.0,229.274100,159.584269,19.92,127.5400,191.090,306.5125,958.71
arson,92.0,24.661957,20.870553,0.73,10.2775,18.370,33.3600,129.55


In [30]:
crime_data_250_plus.isnull().sum()

states           0
cities           0
population       0
total_crime      0
murder           0
rape             4
robbery          0
agrv_assault     0
violent_crime    4
burglary         0
larceny          0
vehicle_theft    0
prop_crime       0
arson            8
dtype: int64

The final dataset includes cities with populations above 250 000 and contains 100 observations. Some inconsistence in columnsnaming were identified and corrected to ensure alignment with other datasets. This dataset represents large metropolitan areas and provides insight into crime patterns in highly populated regions.

#### Combine Crime Dataset

In [31]:
crime_all_data = pd.concat([crime_data_40_60, crime_data_60_100, crime_data_100_250, crime_data_250_plus], ignore_index=True)

In [32]:
crime_all_data.shape

(975, 14)

In [33]:
crime_all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         975 non-null    str    
 1   cities         975 non-null    str    
 2   population     975 non-null    str    
 3   violent_crime  952 non-null    object 
 4   murder         975 non-null    object 
 5   rape           954 non-null    object 
 6   robbery        975 non-null    float64
 7   agrv_assault   973 non-null    object 
 8   prop_crime     973 non-null    object 
 9   burglary       973 non-null    object 
 10  larceny        975 non-null    object 
 11  vehicle_theft  975 non-null    object 
 12  total_crime    100 non-null    float64
 13  arson          92 non-null     float64
dtypes: float64(3), object(8), str(3)
memory usage: 106.8+ KB


In [34]:
crime_all_data.describe().T

,count,mean,std,min,25%,50%,75%,max
robbery,975.0,125.284421,123.387181,0.00,40.5500,84.00,167.865,972.10
total_crime,100.0,4312.471900,1674.275944,1381.31,3029.7800,4091.50,5392.175,8734.98
arson,92.0,24.661957,20.870553,0.73,10.2775,18.37,33.360,129.55


In [35]:
crime_all_data.isnull().sum()

states             0
cities             0
population         0
violent_crime     23
murder             0
rape              21
robbery            0
agrv_assault       2
prop_crime         2
burglary           2
larceny            0
vehicle_theft      0
total_crime      875
arson            883
dtype: int64

To create a unified dateset, the four crime datasets were combined into a single dataset representing cities of varying population sizes. The resulting combined dataset provides a comprehensive overview of crime across different urban environments, enabling more robust analysis when merged with housing and air quality data. This integration allows for the investigation of how crime rates, environmental conditions and population characteristics jointly influence housing prices.

Convert to correct data types:

In [36]:
numeric_cols = ["population", "violent_crime", "murder", "rape", "robbery", "agrv_assault", "prop_crime", "burglary", "larceny", "vehicle_theft", "total_crime"]

In [37]:
def clean_col(col):
    return pd.to_numeric(col.astype(str).str.replace(",", "").replace("-",""), errors="coerce")

In [38]:
for col in numeric_cols:
    if col in crime_all_data.columns:
        crime_all_data[col]=clean_col(crime_all_data[col])

In [39]:
crime_all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         975 non-null    str    
 1   cities         975 non-null    str    
 2   population     975 non-null    int64  
 3   violent_crime  952 non-null    float64
 4   murder         864 non-null    float64
 5   rape           951 non-null    float64
 6   robbery        975 non-null    float64
 7   agrv_assault   973 non-null    float64
 8   prop_crime     973 non-null    float64
 9   burglary       973 non-null    float64
 10  larceny        975 non-null    float64
 11  vehicle_theft  975 non-null    float64
 12  total_crime    100 non-null    float64
 13  arson          92 non-null     float64
dtypes: float64(11), int64(1), str(2)
memory usage: 106.8 KB


A reusable preprocessing function was implemented to standarlize numerical columns across datasets. The function removes formatting inconsistencies such as commas and converts all relevant features into numeric types.

#### State Standartization

In [40]:
crime_all_data.states = crime_all_data.states.apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

In [41]:
crime_all_data[
    crime_all_data.states.str.len() !=2
]

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft,total_crime,arson
905,NaN,"Washington, D.C.",693972,948.74,16.72,63.84,338.77,529.42,4156.22,260.53,3528.96,366.73,5104.96,NaN


In [42]:
crime_all_data.loc[
    (crime_all_data.cities == "Washington, D.C."),
    "states"
] = "DC"

In [43]:
crime_all_data[
    crime_all_data.states.str.len() !=2
]

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft,total_crime,arson


As part of preprocessing stage, the state information in the crime dataset was srandardized to ensure consistency across all datasets. Since the original data contained full state names and in some cases non-standard values, the Python library **us** was used to convert all entries into official two-letter state abbreviations. The conversion was performed using lookup finction which maps each state name to its corresponding abbreviation. If a value could not be match to a valid U.S. state, it was replaced with a missing value(NaN).

After the transformation, additional validation was applied to ensure that all state entries consist of exactly two characters. Any remaining invalid or missing values were identified and corrected manually where possible. For example. entries such as "Washington, D.C." were handled separately to maintain consistency in the dataset. This step ensured that the state column is fully standardized and suitable for merging with other datasets, such as housing and air quality data, which alse rely on state-level identifiers.

### 3.3 Air Quality Data

In [44]:
polution_data = pd.read_csv("data/pollution_us_2000_2016.csv")

In [45]:
polution_data.head(5)

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,...,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI
0,0,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,3.000000,9.0,21,13.0,Parts per million,1.145833,4.2,21,NaN
1,1,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,3.000000,9.0,21,13.0,Parts per million,0.878947,2.2,23,25.0
2,2,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,2.975000,6.6,23,NaN,Parts per million,1.145833,4.2,21,NaN
3,3,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,2.975000,6.6,23,NaN,Parts per million,0.878947,2.2,23,25.0
4,4,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-02,Parts per billion,...,Parts per billion,1.958333,3.0,22,4.0,Parts per million,0.850000,1.6,23,NaN


In [46]:
polution_data.shape

(1746661, 29)

In [47]:
polution_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1746661 entries, 0 to 1746660
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Unnamed: 0         int64  
 1   State Code         int64  
 2   County Code        int64  
 3   Site Num           int64  
 4   Address            str    
 5   State              str    
 6   County             str    
 7   City               str    
 8   Date Local         str    
 9   NO2 Units          str    
 10  NO2 Mean           float64
 11  NO2 1st Max Value  float64
 12  NO2 1st Max Hour   int64  
 13  NO2 AQI            int64  
 14  O3 Units           str    
 15  O3 Mean            float64
 16  O3 1st Max Value   float64
 17  O3 1st Max Hour    int64  
 18  O3 AQI             int64  
 19  SO2 Units          str    
 20  SO2 Mean           float64
 21  SO2 1st Max Value  float64
 22  SO2 1st Max Hour   int64  
 23  SO2 AQI            float64
 24  CO Units           str    
 25  CO Mean            float64
 2

In [48]:
polution_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,1746661.0,54714.136753,33729.076302,0.0000,25753.000000,53045.000000,80336.000000,134575.000000
State Code,1746661.0,22.309065,17.256205,1.0000,6.000000,17.000000,40.000000,80.000000
County Code,1746661.0,71.693807,79.480231,1.0000,17.000000,59.000000,97.000000,650.000000
Site Num,1746661.0,1118.214373,2003.103069,1.0000,9.000000,60.000000,1039.000000,9997.000000
NO2 Mean,1746661.0,12.821930,9.504814,-2.0000,5.750000,10.739130,17.713636,139.541667
NO2 1st Max Value,1746661.0,25.414848,15.999630,-2.0000,13.000000,24.000000,35.700000,267.000000
NO2 1st Max Hour,1746661.0,11.731023,7.877501,0.0000,5.000000,9.000000,20.000000,23.000000
NO2 AQI,1746661.0,23.898217,15.162805,0.0000,12.000000,23.000000,33.000000,132.000000
O3 Mean,1746661.0,0.026125,0.011370,0.0000,0.017875,0.025875,0.033917,0.095083
O3 1st Max Value,1746661.0,0.039203,0.015344,0.0000,0.029000,0.038000,0.048000,0.141000


Descriptive statistics were computed to better understand the distribution of the air quality variables. The results show significant variability in pollutant levels across different locations and time periods, highlighting the importance of incorporating environmental factors into the housing price analysis.

In [49]:
polution_data.isnull().sum()

Unnamed: 0                0
State Code                0
County Code               0
Site Num                  0
Address                   0
State                     0
County                    0
City                      0
Date Local                0
NO2 Units                 0
NO2 Mean                  0
NO2 1st Max Value         0
NO2 1st Max Hour          0
NO2 AQI                   0
O3 Units                  0
O3 Mean                   0
O3 1st Max Value          0
O3 1st Max Hour           0
O3 AQI                    0
SO2 Units                 0
SO2 Mean                  0
SO2 1st Max Value         0
SO2 1st Max Hour          0
SO2 AQI              872907
CO Units                  0
CO Mean                   0
CO 1st Max Value          0
CO 1st Max Hour           0
CO AQI               873323
dtype: int64

The air quality daraset was introduced to capture environmrntal factors that may influence housing prices. The dataset contains detailed measurements of major air pollutants across different locations in the United States over time. The data was loaded from a CSV file and initially explored using standard data inspection techniques such as .head(), .info() and describe(). This allowed for overview of the dataset structure, data types and statistical properties of the variables. The dataset includes information such as state, city, monitoring site identifiers and date of measurement. Additionally, it contains multiple air pollution indicators, includiing nitrogen dioxide(NO2), ozone(O2), sulfur dioxide(SO2) and carbon monoxide(CO) along with their respective mean values, maximum values and Air Quality Index(AQI) measurements. The dataset is relatively large, consisting of over 1.7 million records and 29 columns which makes it suitable for robust statistical analysis. Most pollutant-related variables are already stored in numeric format, facilitating further processing and modeling.

#### State Standartization

In [50]:
polution_data.State = polution_data.State.apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

In [51]:
polution_data[
    polution_data.State.str.len() != 2
]

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,...,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI
39766,39766,11,1,41,"420 34th Street N.E.,Washington, DC 20019",NaN,District of Columbia,Washington,2000-01-01,Parts per billion,...,Parts per billion,11.250000,33.0,15,47.0,Parts per million,2.062500,4.100,3,NaN
39767,39767,11,1,41,"420 34th Street N.E.,Washington, DC 20019",NaN,District of Columbia,Washington,2000-01-01,Parts per billion,...,Parts per billion,11.250000,33.0,15,47.0,Parts per million,2.005263,3.600,5,41.0
39768,39768,11,1,41,"420 34th Street N.E.,Washington, DC 20019",NaN,District of Columbia,Washington,2000-01-01,Parts per billion,...,Parts per billion,11.225000,19.0,17,NaN,Parts per million,2.062500,4.100,3,NaN
39769,39769,11,1,41,"420 34th Street N.E.,Washington, DC 20019",NaN,District of Columbia,Washington,2000-01-01,Parts per billion,...,Parts per billion,11.225000,19.0,17,NaN,Parts per million,2.005263,3.600,5,41.0
39770,39770,11,1,41,"420 34th Street N.E.,Washington, DC 20019",NaN,District of Columbia,Washington,2000-01-02,Parts per billion,...,Parts per billion,7.791667,11.0,0,16.0,Parts per million,0.658333,2.600,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1730836,8779,11,1,43,"2500 1ST STREET, N.W. WASHINGTON DC",NaN,District of Columbia,Washington,2016-04-29,Parts per billion,...,Parts per billion,1.100000,1.7,8,NaN,Parts per million,0.200000,0.200,0,2.0
1730837,8780,11,1,43,"2500 1ST STREET, N.W. WASHINGTON DC",NaN,District of Columbia,Washington,2016-04-30,Parts per billion,...,Parts per billion,0.366667,1.4,9,1.0,Parts per million,0.212250,0.297,21,NaN
1730838,8781,11,1,43,"2500 1ST STREET, N.W. WASHINGTON DC",NaN,District of Columbia,Washington,2016-04-30,Parts per billion,...,Parts per billion,0.366667,1.4,9,1.0,Parts per million,0.208333,0.300,22,3.0
1730839,8782,11,1,43,"2500 1ST STREET, N.W. WASHINGTON DC",NaN,District of Columbia,Washington,2016-04-30,Parts per billion,...,Parts per billion,0.337500,0.9,8,NaN,Parts per million,0.212250,0.297,21,NaN


In [52]:
polution_data.loc[
    (polution_data.City == "Washington"),
    "State"
] = "DC"

In [53]:
polution_data[
    polution_data.State.str.len() != 2
]

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,...,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI
625533,97067,80,2,3,"13891 CAJEME, LA MESA",NaN,BAJA CALIFORNIA NORTE,Tijuana,2006-01-01,Parts per billion,...,Parts per billion,0.260870,1.0,0,1.0,Parts per million,0.756522,2.1,5,NaN
625534,97068,80,2,3,"13891 CAJEME, LA MESA",NaN,BAJA CALIFORNIA NORTE,Tijuana,2006-01-01,Parts per billion,...,Parts per billion,0.260870,1.0,0,1.0,Parts per million,0.722222,1.6,6,18.0
625535,97069,80,2,3,"13891 CAJEME, LA MESA",NaN,BAJA CALIFORNIA NORTE,Tijuana,2006-01-01,Parts per billion,...,Parts per billion,0.185714,1.0,2,NaN,Parts per million,0.756522,2.1,5,NaN
625536,97070,80,2,3,"13891 CAJEME, LA MESA",NaN,BAJA CALIFORNIA NORTE,Tijuana,2006-01-01,Parts per billion,...,Parts per billion,0.185714,1.0,2,NaN,Parts per million,0.722222,1.6,6,18.0
625537,97071,80,2,3,"13891 CAJEME, LA MESA",NaN,BAJA CALIFORNIA NORTE,Tijuana,2006-01-02,Parts per billion,...,Parts per billion,0.130435,1.0,21,1.0,Parts per million,0.591304,2.6,23,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1201625,129243,80,2,12,"UABC, CALZADA BENITO JUAREZ, MEXICALI",NaN,BAJA CALIFORNIA NORTE,Mexicali,2011-02-13,Parts per billion,...,Parts per billion,0.057143,0.1,2,NaN,Parts per million,2.087500,3.5,0,40.0
1201626,129244,80,2,12,"UABC, CALZADA BENITO JUAREZ, MEXICALI",NaN,BAJA CALIFORNIA NORTE,Mexicali,2011-02-14,Parts per billion,...,Parts per billion,0.055000,0.1,0,0.0,Parts per million,2.309091,5.1,1,NaN
1201627,129245,80,2,12,"UABC, CALZADA BENITO JUAREZ, MEXICALI",NaN,BAJA CALIFORNIA NORTE,Mexicali,2011-02-14,Parts per billion,...,Parts per billion,0.055000,0.1,0,0.0,Parts per million,2.616667,4.6,2,52.0
1201628,129246,80,2,12,"UABC, CALZADA BENITO JUAREZ, MEXICALI",NaN,BAJA CALIFORNIA NORTE,Mexicali,2011-02-14,Parts per billion,...,Parts per billion,0.020000,0.1,2,NaN,Parts per million,2.309091,5.1,1,NaN


In [54]:
polution_data = polution_data[
    ~polution_data.City.isin(["Tijuana", "Mexicali", "Rosarito"])
]

In [55]:
polution_data[
    polution_data.State.str.len() != 2
]

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,...,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI


As part of the processing of the air quality dataset, the stae information was standardized to ensure consistency across all datasets used in the analysis. Initially, full state names were converted into official two-letter abbreviations using the **us** Python library. Any values that could not be matched to a valid U.S. state were replaced with missing values(NaN). Following this transformation validation checks were performed to identify entries that did not conform to the expected two-character state format. This inconsistencies revealed special cases such as "Washighton, D.C." where the state value could not be automatically corrected by assigningthe appropriate abbreviation("DC"). Additionally, records corresponding to non-U.S. locations (e.g. cities such as Tijuana, Mexicali and Rosarito) were identified. Since these entries fall outside the scope of the analysis which focuses on U.S. housing markets, they were excluded from the dataset.

## 4. Data Aggregation and Integration

### 4.1 Data Aggregation

After cleaning and standardizing the datasets, the next step is to aggregate the external data at the state level in order to make it compatible with the housing dataset. Since the housing data is organized using state-level geographical information, both the crime and air quality datasets will be transformed into aggregated state-level erpresentations.

For the crime data, multiple datasets corresponding to different city population sizes will be grouped by state. Key crime indicators such as violent crime, murder, robbery, assult, burglary and theft will be aggregated using mean values providing a representative estimate of crime levels for each state. 

In [56]:
crime_all_data = crime_all_data.rename(columns={"states" : "State"})

In [57]:
crime_grouped = crime_all_data.groupby(["State"], as_index=False)[["violent_crime", "murder", "rape", "robbery","agrv_assault","prop_crime", "burglary", "larceny", "vehicle_theft", "total_crime", "arson"]].mean()

In [58]:
crime_grouped

,State,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft,total_crime,arson
0,AK,1203.290000,9.120000,132.010000,262.670000,799.490000,5415.820000,748.170000,3619.660000,1047.980000,6640.040000,20.930000
1,AL,555.625000,10.353000,33.836000,173.111000,338.405000,4612.973000,1205.664000,3122.411000,284.908000,6217.020000,22.940000
2,AR,696.844444,8.933333,59.788889,159.044444,469.077778,5465.711111,1323.100000,3862.866667,279.755556,NaN,NaN
3,AZ,304.141154,4.597500,27.905769,78.177692,193.813462,3237.886154,612.743077,2419.731538,205.418846,3509.647143,14.386667
4,CA,355.028107,3.991034,20.036990,122.196845,209.424223,2697.687767,617.547767,1683.247427,396.889272,3626.465000,28.005556
5,CO,318.877500,4.270625,45.089500,61.958000,208.418000,2942.870000,490.377500,2207.228500,245.258500,3918.903333,20.353333
6,CT,346.880000,4.300000,26.288000,131.988000,184.824000,2605.252000,504.112000,1869.292000,231.860000,NaN,NaN
7,DC,948.740000,16.720000,63.840000,338.770000,529.420000,4156.220000,260.530000,3528.960000,366.730000,5104.960000,NaN
8,DE,1703.500000,36.100000,47.200000,657.500000,962.700000,5304.600000,1681.300000,3117.000000,506.300000,NaN,NaN
9,FL,566.172923,6.244687,32.676308,176.347231,350.997692,4029.814154,904.506615,2882.568154,242.747077,4132.621667,12.111667


For the air quality data, pollutant measurements such as NO2, O3, SO2 and CO will be grouped by state and summary statisrtics such as mean, max and standard deviation will be calculated for each pollutant. This approach captures not only the average pollution levels but also the variability and extreme values within each state.

In [59]:
air_cols = [
    "State",
    "NO2 Mean",
    "O3 Mean",
    "SO2 Mean",
    "CO Mean"
]

In [60]:
air_features = polution_data[air_cols].groupby("State").agg({
    "NO2 Mean": ["mean", "max", "std"],
    "O3 Mean": ["mean", "max", "std"],
    "SO2 Mean": ["mean", "max", "std"],
    "CO Mean": ["mean", "max", "std"]
}).reset_index()

In [61]:
air_features.columns = [
    "State",
    "NO2_mean", "NO2_max", "NO2_std",
    "O3_mean", "O3_max", "O3_std",
    "SO2_mean", "SO2_max", "SO2_std",
    "CO_mean", "CO_max", "CO_std",
]

In [62]:
air_features

,State,NO2_mean,NO2_max,NO2_std,O3_mean,O3_max,O3_std,SO2_mean,SO2_max,SO2_std,CO_mean,CO_max,CO_std
0,AK,11.313152,49.795833,9.478902,0.012799,0.039042,0.008542,6.083755,31.525000,4.823566,0.423438,2.032917,0.285521
1,AL,9.410693,58.581818,4.573633,0.024612,0.045792,0.008218,1.034236,9.100000,1.186772,0.212607,0.705556,0.085012
2,AR,9.753701,33.675000,4.817515,0.026169,0.062667,0.009105,1.383302,10.433333,0.750594,0.422393,1.591667,0.170145
3,AZ,19.067975,139.541667,9.803850,0.024989,0.063167,0.011060,1.364213,19.375000,1.222111,0.490616,3.575000,0.349190
4,CA,13.651894,98.130435,11.081922,0.026053,0.086214,0.011530,1.146695,40.271429,1.406326,0.447485,7.508333,0.375866
5,CO,19.634275,92.000000,11.541242,0.023551,0.061875,0.011626,1.507634,21.066667,1.584635,0.443112,2.475000,0.255458
6,CT,8.990965,53.375000,7.425057,0.028920,0.068542,0.010752,0.912121,28.526316,1.567965,0.250866,1.250000,0.129455
7,DC,17.689366,65.208333,8.304517,0.024184,0.070542,0.011702,4.209670,42.916667,3.568701,0.790170,3.530435,0.491327
8,DE,11.584773,36.541667,6.279272,0.026494,0.061917,0.012188,1.014242,4.575000,0.734727,0.261600,0.850000,0.101410
9,FL,7.363386,42.300000,4.895178,0.026799,0.062958,0.009737,0.495490,13.791667,1.005301,0.425255,1.991667,0.233215


### 4.2 Data Integration

After aggregating the external datasets, the next step is to integrate them with the housing dataset to create a unified analytical framework. The is achieved by merging the datasets based on the common geographical identifier, namely the state abbreviation. 
The housing dataset already contained state-level information extracted during preprocessing while both the crime and air quality datasets had been transformed into state-level summaries. This ensured compatibility across all datasets and allowed for a straightfoward merge operation.

The merging process is performed using a left join, preserving all records from the housing dataset while enriching them with additional features from the crime and air quality data. 

In [63]:
merged_housing_crime_data = housing_data.merge(crime_grouped, left_on=["State"], right_on=["State"], how="left")

In [64]:
merged_housing_crime_data.head(5)

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address,State,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft,total_crime,arson
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701...",NE,360.140000,2.933333,70.213333,68.893333,218.136667,3579.590000,501.576667,2702.733333,375.280000,4527.490000,NaN
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA...",CA,355.028107,3.991034,20.036990,122.196845,209.424223,2697.687767,617.547767,1683.247427,396.889272,3626.465000,28.005556
2,61287.067179,5.865890,8.512727,5.13,36882.159400,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482...",WI,409.033333,4.569167,38.663333,113.007333,253.708000,3219.590000,630.770000,2431.235333,157.577333,4241.225000,28.250000
3,80175.754159,4.988408,6.104512,4.04,26748.428425,1.068138e+06,"06039 Jennifer Islands Apt. 443\nTracyport, KS...",KS,414.619000,5.478571,43.268000,76.066000,291.460000,3909.994000,678.555000,2891.567000,339.872000,6583.500000,25.570000
4,64698.463428,6.025336,8.147760,3.41,60828.249085,1.502056e+06,"4759 Daniel Shoals Suite 442\nNguyenburgh, CO ...",CO,318.877500,4.270625,45.089500,61.958000,208.418000,2942.870000,490.377500,2207.228500,245.258500,3918.903333,20.353333


Following the merge, the resulting dtaset contained a comprehensive set of features, including property characteristics, crime statistics and environmental indicators. However, due to differenes in data availability across states and datasets, some missing values were introduced, particulary in crime and air quality variables.

In [65]:
final_merged_data = merged_housing_crime_data.merge(air_features, on="State", how="left")

In [66]:
final_merged_data.head(5)

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address,State,violent_crime,murder,...,NO2_std,O3_mean,O3_max,O3_std,SO2_mean,SO2_max,SO2_std,CO_mean,CO_max,CO_std
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701...",NE,360.140000,2.933333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA...",CA,355.028107,3.991034,...,11.081922,0.026053,0.086214,0.011530,1.146695,40.271429,1.406326,0.447485,7.508333,0.375866
2,61287.067179,5.865890,8.512727,5.13,36882.159400,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482...",WI,409.033333,4.569167,...,6.187916,0.022432,0.054708,0.011567,2.614880,11.541667,1.247223,0.350427,1.008333,0.092342
3,80175.754159,4.988408,6.104512,4.04,26748.428425,1.068138e+06,"06039 Jennifer Islands Apt. 443\nTracyport, KS...",KS,414.619000,5.478571,...,7.701917,0.026002,0.071458,0.011581,2.368248,25.875000,2.436746,0.408664,2.583333,0.315996
4,64698.463428,6.025336,8.147760,3.41,60828.249085,1.502056e+06,"4759 Daniel Shoals Suite 442\nNguyenburgh, CO ...",CO,318.877500,4.270625,...,11.541242,0.023551,0.061875,0.011626,1.507634,21.066667,1.584635,0.443112,2.475000,0.255458


In [67]:
final_merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3816 entries, 0 to 3815
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Avg. Area Income              3816 non-null   float64
 1   Avg. Area House Age           3816 non-null   float64
 2   Avg. Area Number of Rooms     3816 non-null   float64
 3   Avg. Area Number of Bedrooms  3816 non-null   float64
 4   Area Population               3816 non-null   float64
 5   Price                         3816 non-null   float64
 6   Address                       3816 non-null   str    
 7   State                         3816 non-null   str    
 8   violent_crime                 3649 non-null   float64
 9   murder                        3649 non-null   float64
 10  rape                          3649 non-null   float64
 11  robbery                       3649 non-null   float64
 12  agrv_assault                  3649 non-null   float64
 13  prop_crime    

In [68]:
final_merged_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Avg. Area Income,3816.0,6.861121e+04,10663.226533,17796.631190,61521.223419,6.879470e+04,7.586708e+04,1.077017e+05
Avg. Area House Age,3816.0,5.984038e+00,0.989588,2.644304,5.331168,5.980470e+00,6.658717e+00,9.519088e+00
Avg. Area Number of Rooms,3816.0,6.972231e+00,1.001500,3.236194,6.289761,6.990750e+00,7.649015e+00,1.075959e+01
Avg. Area Number of Bedrooms,3816.0,3.968391e+00,1.231785,2.000000,3.130000,4.040000e+00,4.480000e+00,6.500000e+00
Area Population,3816.0,3.607186e+04,9928.419826,172.610686,29288.481593,3.623948e+04,4.278623e+04,6.959204e+04
Price,3816.0,1.229275e+06,349826.162977,15938.657923,997160.953673,1.231482e+06,1.463190e+06,2.370231e+06
violent_crime,3649.0,4.859466e+02,269.062603,190.500000,335.616667,4.146190e+02,5.533133e+02,1.703500e+03
murder,3649.0,6.568154e+00,5.927285,0.450000,2.933333,5.816957e+00,7.932500e+00,3.610000e+01
rape,3649.0,4.292221e+01,22.586314,12.998077,28.942857,3.730500e+01,4.904000e+01,1.320100e+02
robbery,3649.0,1.356025e+02,102.472025,26.044444,78.177692,1.206719e+02,1.577383e+02,6.575000e+02


In [69]:
final_merged_data.isnull().sum()

Avg. Area Income                   0
Avg. Area House Age                0
Avg. Area Number of Rooms          0
Avg. Area Number of Bedrooms       0
Area Population                    0
Price                              0
Address                            0
State                              0
violent_crime                    167
murder                           167
rape                             167
robbery                          167
agrv_assault                     167
prop_crime                       167
burglary                         167
larceny                          167
vehicle_theft                    167
total_crime                     1149
arson                           1389
NO2_mean                         393
NO2_max                          393
NO2_std                          393
O3_mean                          393
O3_max                           393
O3_std                           393
SO2_mean                         393
SO2_max                          393
S

### 4.3 Handling Missing Values

Following the data integration step, a detailed analysis of missing values was performed to asses data completeness across all features. The results showed that while the core housing variables contained no missing values, several features from the crime and air quality datasets exhibited incomplete coverage.

In particular, air quality variables such as NO2, SO2, O3 and CO contained a significant number of missing values, as these measurements were not available for all states. Similarly, certain crime-related features including total crime and arson also had partial data availability.

In [70]:
final_merged_data[final_merged_data.NO2_mean.isna()]

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address,State,violent_crime,murder,...,NO2_std,O3_mean,O3_max,O3_std,SO2_mean,SO2_max,SO2_std,CO_mean,CO_max,CO_std
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701...",NE,360.14,2.933333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,39033.809237,7.671755,7.250029,3.10,39220.361467,1.042814e+06,"209 Natasha Stream Suite 961\nHuffmanland, NE ...",NE,360.14,2.933333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,78699.515103,5.652784,6.756454,3.01,22836.607569,1.081150e+06,"7585 Lynn Loop\nEast Judy, WV 73336",WV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,46147.053056,6.623333,7.606832,4.43,27161.128608,8.820572e+05,"90634 Michelle Valleys\nNorth Victoria, WV 99370",WV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,73946.851074,4.863154,7.537182,6.35,35261.127018,1.109588e+06,"8034 Pierce Prairie Suite 727\nDevonfurt, NE 3...",NE,360.14,2.933333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3765,55250.110364,5.560996,6.105254,4.12,22912.066920,5.830168e+05,"216 Erin Junctions Apt. 524\nEast Anthony, VT ...",VT,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3773,68895.754942,5.817512,7.605602,4.24,43119.312679,1.213352e+06,"03733 Spencer Harbor Suite 960\nSouth David, W...",WV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3774,62968.650709,5.293690,5.457162,3.25,24554.562521,5.568396e+05,"02689 George Hill\nLake Chadton, MS 30265",MS,495.26,14.580000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3790,74941.585801,7.453375,7.308749,6.01,28977.941178,1.626369e+06,2480 Catherine Cliffs Suite 880\nCynthiacheste...,MS,495.26,14.580000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


To ensure consistency and reliability in the modeling process, rows with missing values in key environmental indicators were removed. This step ensured that all remaining observations contained complete air quality information.

In [71]:
final_merged_data = final_merged_data.dropna(subset="NO2_mean")

In [72]:
final_merged_data.isnull().sum()

Avg. Area Income                  0
Avg. Area House Age               0
Avg. Area Number of Rooms         0
Avg. Area Number of Bedrooms      0
Area Population                   0
Price                             0
Address                           0
State                             0
violent_crime                     0
murder                            0
rape                              0
robbery                           0
agrv_assault                      0
prop_crime                        0
burglary                          0
larceny                           0
vehicle_theft                     0
total_crime                     840
arson                           996
NO2_mean                          0
NO2_max                           0
NO2_std                           0
O3_mean                           0
O3_max                            0
O3_std                            0
SO2_mean                          0
SO2_max                           0
SO2_std                     

Additional columns with high proportion of missing values such as total_crime and arson, are removed from the feature dataset to avoid introducing noise or bias into the model.

In [73]:
columns_to_drop = ["total_crime", "arson"]

In [74]:
final_merged_data = final_merged_data.drop(columns=columns_to_drop)

In [75]:
columns_to_drop = "Price"

In [76]:
X = final_merged_data.drop(columns=columns_to_drop)
y = final_merged_data.Price


In [77]:
X.isnull().sum()

Avg. Area Income                0
Avg. Area House Age             0
Avg. Area Number of Rooms       0
Avg. Area Number of Bedrooms    0
Area Population                 0
Address                         0
State                           0
violent_crime                   0
murder                          0
rape                            0
robbery                         0
agrv_assault                    0
prop_crime                      0
burglary                        0
larceny                         0
vehicle_theft                   0
NO2_mean                        0
NO2_max                         0
NO2_std                         0
O3_mean                         0
O3_max                          0
O3_std                          0
SO2_mean                        0
SO2_max                         0
SO2_std                         0
CO_mean                         0
CO_max                          0
CO_std                          0
dtype: int64